# Embed Markdown to Pinecone via OpenRouter

This notebook loads Markdown files from `data/knowledge_base`, builds section-aware chunks, creates embeddings with OpenRouter using `openai/text-embedding-3-large`, and upserts them into the Pinecone index `fahmai-text-embedding-3-large`.

The target Pinecone index is expected to be `1024` dimensions, so this notebook requests `dimensions=1024` during embedding.


In [1]:
from __future__ import annotations

import hashlib
import math
import os
import re
from collections import OrderedDict
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from markdown_it import MarkdownIt
from openai import OpenAI
from pinecone import Pinecone

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)


In [2]:
INDEX_NAME = "fahmai-text-embedding-3-large"
EMBEDDING_MODEL = "openai/text-embedding-3-large"
EMBEDDING_DIMENSIONS = 1024
CHUNK_SIZE = 450
CHUNK_OVERLAP = 75
EMBED_BATCH_SIZE = 32
UPSERT_BATCH_SIZE = 100
MAX_FILES = None
MAX_CHUNKS = None
NAMESPACE = "knowledge-base-markdown"
TOP_K = 5


In [3]:
def load_env_from_parents(start: Path | None = None) -> Path | None:
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        env_path = base / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


def resolve_project_dir() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "knowledge_base").exists() and (
            candidate / "pyproject.toml"
        ).exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the competition project directory from the current working directory."
    )


env_path = load_env_from_parents()
project_dir = resolve_project_dir()
kb_dir = project_dir / "data" / "knowledge_base"

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")
site_url = os.getenv("OPENROUTER_SITE_URL")
site_title = os.getenv("OPENROUTER_SITE_TITLE", "FahMai RAG Notebook")

missing = [
    name
    for name, value in {
        "OPENROUTER_API_KEY": openrouter_api_key,
        "PINECONE_API_KEY": pinecone_api_key,
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

print(f"Loaded .env from: {env_path}" if env_path else "Using shell environment only.")
print(f"Project directory: {project_dir}")
print(f"Knowledge base directory: {kb_dir}")


Loaded .env from: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/.env
Project directory: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1
Knowledge base directory: /Users/beam/Workspace/Project/my-kaggle-notebooks/notebooks/2026-03-27-super-ai-engineer-s6-fahmai-rag-challenge-level-1/data/knowledge_base


In [4]:
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key
)
pc = Pinecone(api_key=pinecone_api_key)

index_description = pc.describe_index(INDEX_NAME)
index_info = (
    index_description.to_dict()
    if hasattr(index_description, "to_dict")
    else dict(index_description)
)
dimension = index_info.get("dimension") or index_info.get("spec", {}).get("dimension")
metric = index_info.get("metric")
host = index_info.get("host")

if dimension != EMBEDDING_DIMENSIONS:
    raise RuntimeError(
        f"Index {INDEX_NAME!r} has dimension {dimension}, expected {EMBEDDING_DIMENSIONS}."
    )

index = pc.Index(INDEX_NAME)
display(
    pd.DataFrame(
        [
            {
                "index_name": INDEX_NAME,
                "dimension": dimension,
                "metric": metric,
                "host": host,
            }
        ]
    )
)


,index_name,dimension,metric,host
0,fahmai-text-embedding-3-large,1024,cosine,fahmai-text-embedding-3-large-ctitniy.svc.aped-4627-b74a.pinecone.io


In [5]:
def normalize_whitespace(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


try:
    import tiktoken
except ImportError:
    tiktoken = None


def estimate_token_count(text: str) -> int:
    cleaned = normalize_whitespace(text)
    if not cleaned:
        return 0
    if tiktoken is not None:
        encoder = tiktoken.get_encoding("cl100k_base")
        return len(encoder.encode(cleaned))
    return max(1, math.ceil(len(cleaned) / 4))


def render_heading_lines(path: list[str], start_level: int = 1) -> str:
    if not path:
        return ""
    return "\n".join(
        f"{'#' * (start_level + idx)} {heading}" for idx, heading in enumerate(path)
    )


def dedupe_block_texts(block_texts: list[str]) -> list[str]:
    deduped: list[str] = []
    for text in block_texts:
        normalized = normalize_whitespace(text)
        if not normalized:
            continue
        if any(
            normalized == existing or normalized in existing for existing in deduped
        ):
            continue
        deduped = [existing for existing in deduped if existing not in normalized]
        deduped.append(normalized)
    return deduped


def compose_section_text(heading_path: list[str], block_texts: list[str]) -> str:
    pieces = []
    heading_lines = render_heading_lines(heading_path)
    if heading_lines:
        pieces.append(heading_lines)
    pieces.extend(dedupe_block_texts(block_texts))
    return normalize_whitespace("\n\n".join(pieces))


def parse_markdown_blocks(text: str, source_path: str) -> list[dict[str, Any]]:
    md = MarkdownIt("commonmark").enable("table").enable("strikethrough")
    tokens = md.parse(text)
    lines = text.splitlines()
    stack: list[str | None] = []
    blocks: list[dict[str, Any]] = []

    def current_path() -> list[str]:
        return [item for item in stack if item]

    i = 0
    while i < len(tokens):
        token = tokens[i]
        if token.type == "heading_open":
            level = int(token.tag[1])
            inline_token = tokens[i + 1] if i + 1 < len(tokens) else None
            heading_text = normalize_whitespace(getattr(inline_token, "content", ""))
            while len(stack) < level:
                stack.append(None)
            stack[level - 1] = heading_text
            stack[:] = stack[:level]
            i += 1
            continue

        if token.type in {
            "paragraph_open",
            "bullet_list_open",
            "ordered_list_open",
            "table_open",
        }:
            start_line, end_line = token.map or (None, None)
            segment = (
                "\n".join(lines[start_line:end_line])
                if start_line is not None and end_line is not None
                else ""
            )
            segment = normalize_whitespace(segment)
            if segment:
                blocks.append(
                    {
                        "source_path": source_path,
                        "block_type": token.type.removesuffix("_open"),
                        "heading_path": current_path(),
                        "text": segment,
                    }
                )
        elif token.type == "fence" and token.content:
            fence_text = normalize_whitespace(token.markup + "\n" + token.content)
            blocks.append(
                {
                    "source_path": source_path,
                    "block_type": "fence",
                    "heading_path": current_path(),
                    "text": fence_text,
                }
            )
        i += 1
    return blocks


def build_sections(
    blocks: list[dict[str, Any]], merge_preamble_tokens: int = 80
) -> list[dict[str, Any]]:
    grouped: OrderedDict[tuple[str, ...], list[dict[str, Any]]] = OrderedDict()
    for block in blocks:
        key = tuple(block["heading_path"])
        grouped.setdefault(key, []).append(block)

    sections: list[dict[str, Any]] = []
    for key, grouped_blocks in grouped.items():
        block_texts = [block["text"] for block in grouped_blocks]
        section_text = compose_section_text(list(key), block_texts)
        sections.append(
            {
                "source_path": grouped_blocks[0]["source_path"],
                "heading_path": list(key),
                "block_count": len(grouped_blocks),
                "text": section_text,
                "estimated_tokens": estimate_token_count(section_text),
            }
        )

    if len(sections) >= 2:
        first_section = sections[0]
        second_section = sections[1]
        if (
            len(first_section["heading_path"]) == 1
            and len(second_section["heading_path"]) >= 2
            and second_section["heading_path"][0] == first_section["heading_path"][0]
            and first_section["estimated_tokens"] <= merge_preamble_tokens
        ):
            merged_block_texts = [first_section["text"]]
            child_heading_lines = render_heading_lines(
                second_section["heading_path"][1:], start_level=2
            )
            if child_heading_lines:
                merged_block_texts.append(child_heading_lines)
            merged_block_texts.append(
                "\n\n".join(
                    block["text"]
                    for block in grouped[tuple(second_section["heading_path"])]
                )
            )
            sections[1] = {
                **second_section,
                "block_count": first_section["block_count"]
                + second_section["block_count"],
                "text": normalize_whitespace(
                    "\n\n".join(part for part in merged_block_texts if part)
                ),
            }
            sections[1]["estimated_tokens"] = estimate_token_count(sections[1]["text"])
            sections = sections[1:]
    return sections


def split_text_with_overlap(
    text: str, chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP
) -> list[str]:
    text = normalize_whitespace(text)
    if estimate_token_count(text) <= chunk_size:
        return [text]

    separators = ["\n\n", "\n", ". ", " "]
    pieces = [text]
    for separator in separators:
        if all(estimate_token_count(piece) <= chunk_size for piece in pieces):
            break
        next_pieces: list[str] = []
        for piece in pieces:
            if estimate_token_count(piece) <= chunk_size or separator not in piece:
                next_pieces.append(piece)
                continue
            split_parts = [
                normalize_whitespace(part)
                for part in piece.split(separator)
                if normalize_whitespace(part)
            ]
            if separator in {". ", " "}:
                suffix = "." if separator == ". " else ""
                split_parts = [
                    part + suffix if suffix and not part.endswith(suffix) else part
                    for part in split_parts
                ]
            next_pieces.extend(split_parts)
        pieces = next_pieces

    hard_limit = max(200, chunk_size * 4)
    flattened: list[str] = []
    for piece in pieces:
        if estimate_token_count(piece) <= chunk_size:
            flattened.append(piece)
            continue
        for start in range(0, len(piece), hard_limit):
            flattened.append(piece[start : start + hard_limit])

    chunks: list[str] = []
    current = ""
    overlap_chars = max(80, chunk_overlap * 6)
    for piece in flattened:
        piece = normalize_whitespace(piece)
        if not piece:
            continue
        candidate = normalize_whitespace(f"{current}\n\n{piece}" if current else piece)
        if current and estimate_token_count(candidate) > chunk_size:
            chunks.append(current)
            overlap_seed = current[-overlap_chars:]
            current = normalize_whitespace(f"{overlap_seed}\n\n{piece}")
            if estimate_token_count(current) > chunk_size:
                chunks.append(piece)
                current = ""
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks


def build_rag_records(
    sections: list[dict[str, Any]],
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    next_chunk_number = 0
    for section_index, section in enumerate(sections):
        child_chunks = split_text_with_overlap(
            section["text"], chunk_size=chunk_size, chunk_overlap=chunk_overlap
        )
        for chunk_in_section, chunk_text in enumerate(child_chunks):
            chunk_id = f"{section['source_path']}#chunk_{next_chunk_number:04d}"
            content_hash = hashlib.sha256(chunk_text.encode("utf-8")).hexdigest()
            records.append(
                {
                    "id": chunk_id,
                    "chunk_text": chunk_text,
                    "chunk_tokens": estimate_token_count(chunk_text),
                    "metadata": {
                        "content_hash": content_hash,
                        "document_id": section["source_path"],
                        "file_path": section["source_path"],
                        "heading_path": section["heading_path"],
                        "section_index": section_index,
                        "chunk_number": next_chunk_number,
                        "chunk_in_section": chunk_in_section,
                        "source_type": "markdown",
                        "text_preview": chunk_text[:200],
                    },
                }
            )
            next_chunk_number += 1
    return records


def load_markdown_corpus(
    root: Path, max_files: int | None = None
) -> list[dict[str, Any]]:
    paths = sorted(root.rglob("*.md"))
    if max_files is not None:
        paths = paths[:max_files]
    records = []
    for path in paths:
        records.append(
            {
                "source_path": str(path.relative_to(root.parent)),
                "path": path,
                "text": path.read_text(encoding="utf-8"),
            }
        )
    return records


def build_all_chunks(
    root: Path, max_files: int | None = None, max_chunks: int | None = None
) -> list[dict[str, Any]]:
    all_records: list[dict[str, Any]] = []
    for doc in load_markdown_corpus(root, max_files=max_files):
        blocks = parse_markdown_blocks(doc["text"], doc["source_path"])
        sections = build_sections(blocks)
        all_records.extend(build_rag_records(sections))
    if max_chunks is not None:
        return all_records[:max_chunks]
    return all_records


In [6]:
chunk_records = build_all_chunks(kb_dir, max_files=MAX_FILES, max_chunks=MAX_CHUNKS)

preview_df = pd.DataFrame(
    {
        "id": [record["id"] for record in chunk_records],
        "file_path": [record["metadata"]["file_path"] for record in chunk_records],
        "heading_path": [
            " > ".join(record["metadata"]["heading_path"]) for record in chunk_records
        ],
        "chunk_tokens": [record["chunk_tokens"] for record in chunk_records],
        "preview": [record["chunk_text"][:180] for record in chunk_records],
    }
)

print(f"Chunk records prepared: {len(chunk_records)}")
display(preview_df.head(20))


Chunk records prepared: 753


,id,file_path,heading_path,chunk_tokens,preview
0,knowledge_base/policies/cancellation_policy.md#chunk_0000,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 1. ภาพรวมนโยบาย,76,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n\n**วันที่อัปเดต:** 1 มีนาคม 2569\n\n## 1. ภาพรวมนโยบาย\n\nฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื...
1,knowledge_base/policies/cancellation_policy.md#chunk_0001,knowledge_base/policies/cancellation_policy.md,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)",119,"# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 2. การยกเลิกตามสถานะคำสั่งซื้อ\n### 2.1 สถานะ ""รอชำระเงิน"" (Pending Payment)\n\n**ยกเลิกได้ทันที**\n\nคำสั่งซื..."
2,knowledge_base/policies/cancellation_policy.md#chunk_0002,knowledge_base/policies/cancellation_policy.md,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.2 สถานะ ""ชำระเงินแล้ว — กำลังเตรียมจัดส่ง"" (Processing / Preparing to Ship)",164,"# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 2. การยกเลิกตามสถานะคำสั่งซื้อ\n### 2.2 สถานะ ""ชำระเงินแล้ว — กำลังเตรียมจัดส่ง"" (Processing / Preparing to Sh..."
3,knowledge_base/policies/cancellation_policy.md#chunk_0003,knowledge_base/policies/cancellation_policy.md,"นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 2. การยกเลิกตามสถานะคำสั่งซื้อ > 2.3 สถานะ ""จัดส่งแล้ว"" (Shipped / In Transit)",118,"# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 2. การยกเลิกตามสถานะคำสั่งซื้อ\n### 2.3 สถานะ ""จัดส่งแล้ว"" (Shipped / In Transit)\n\n**ไม่สามารถยกเลิกได้**\n\..."
4,knowledge_base/policies/cancellation_policy.md#chunk_0004,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order,56,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n\nสินค้า Pre-order (สั่งจองล่วงหน้า) มีเงื่อนไขการยกเลิกที่แตกต่างจากสินค้...
5,knowledge_base/policies/cancellation_policy.md#chunk_0005,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.1 ยกเลิกก่อน 3 วันหรือมากกว่า ก่อนวันจัดส่ง — **ฟรี ไม่มีค่าใช้จ่าย**,109,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n### 3.1 ยกเลิกก่อน 3 วันหรือมากกว่า ก่อนวันจัดส่ง — **ฟรี ไม่มีค่าใช้จ่าย*...
6,knowledge_base/policies/cancellation_policy.md#chunk_0006,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.2 ยกเลิกน้อยกว่า 3 วัน ก่อนวันจัดส่ง — **หักค่าดำเนินการ 5%**,114,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n### 3.2 ยกเลิกน้อยกว่า 3 วัน ก่อนวันจัดส่ง — **หักค่าดำเนินการ 5%**\n\nหาก...
7,knowledge_base/policies/cancellation_policy.md#chunk_0007,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 3. นโยบายการยกเลิกสินค้า Pre-order > 3.3 หลังวันจัดส่ง — ถือเป็นการคืนสินค้า,97,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 3. นโยบายการยกเลิกสินค้า Pre-order\n### 3.3 หลังวันจัดส่ง — ถือเป็นการคืนสินค้า\n\nหากคำสั่งซื้อ Pre-order ผ่า...
8,knowledge_base/policies/cancellation_policy.md#chunk_0008,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation),50,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation)\n\nในบางกรณี ฟ้าใหม่อาจต้องยกเลิกคำสั่งซื้อของลูกค้าโดย...
9,knowledge_base/policies/cancellation_policy.md#chunk_0009,knowledge_base/policies/cancellation_policy.md,นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่ > 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation) > 4.1 เหตุผลที่ฟ้าใหม่อาจยกเลิกคำสั่งซื้อ,131,# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่\n## 4. การยกเลิกโดยฟ้าใหม่ (FahMai-Initiated Cancellation)\n### 4.1 เหตุผลที่ฟ้าใหม่อาจยกเลิกคำสั่งซื้อ\n\n| สาเหต...


In [7]:
def fetch_existing_hashes(
    record_ids: list[str], namespace: str
) -> dict[str, str | None]:
    existing_hashes: dict[str, str | None] = {}
    for start in range(0, len(record_ids), UPSERT_BATCH_SIZE):
        batch_ids = record_ids[start : start + UPSERT_BATCH_SIZE]
        response = index.fetch(ids=batch_ids, namespace=namespace)
        for record_id in batch_ids:
            vector = response.vectors.get(record_id)
            metadata = vector.metadata if vector else {}
            existing_hashes[record_id] = (
                metadata.get("content_hash") if metadata else None
            )
    return existing_hashes


existing_hashes = fetch_existing_hashes(
    [record["id"] for record in chunk_records], namespace=NAMESPACE
)
pending_chunk_records = [
    record
    for record in chunk_records
    if existing_hashes.get(record["id"]) != record["metadata"]["content_hash"]
]
skipped_chunk_records = len(chunk_records) - len(pending_chunk_records)

print(f"Chunks needing upsert: {len(pending_chunk_records)}")
print(f"Unchanged chunks skipped: {skipped_chunk_records}")


def embed_text_batch(texts: list[str]) -> list[list[float]]:
    headers = {"X-OpenRouter-Title": site_title}
    if site_url:
        headers["HTTP-Referer"] = site_url
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
        encoding_format="float",
        dimensions=EMBEDDING_DIMENSIONS,
        extra_headers=headers,
    )
    vectors = [item.embedding for item in response.data]
    if len(vectors) != len(texts):
        raise RuntimeError(
            f"Expected {len(texts)} embeddings, got {len(vectors)} from {EMBEDDING_MODEL}."
        )
    for idx, vector in enumerate(vectors):
        if len(vector) != EMBEDDING_DIMENSIONS:
            raise RuntimeError(
                f"Embedding dimension mismatch at batch item {idx}: expected {EMBEDDING_DIMENSIONS}, got {len(vector)}."
            )
    return vectors


embedded_records = []
for start in range(0, len(pending_chunk_records), EMBED_BATCH_SIZE):
    batch = pending_chunk_records[start : start + EMBED_BATCH_SIZE]
    vectors = embed_text_batch([record["chunk_text"] for record in batch])
    for record, vector in zip(batch, vectors, strict=True):
        embedded_records.append(
            {
                "id": record["id"],
                "values": vector,
                "metadata": record["metadata"],
            }
        )

if embedded_records:
    print(f"Embedded {len(embedded_records)} chunks.")
    print(f"Vector dimension: {len(embedded_records[0]['values'])}")
else:
    print("No chunks to embed.")


Chunks needing upsert: 753
Unchanged chunks skipped: 0
Embedded 753 chunks.
Vector dimension: 1024


In [8]:
for start in range(0, len(embedded_records), UPSERT_BATCH_SIZE):
    batch = embedded_records[start : start + UPSERT_BATCH_SIZE]
    index.upsert(vectors=batch, namespace=NAMESPACE)

print(
    f"Upserted {len(embedded_records)} vectors into {INDEX_NAME!r} namespace {NAMESPACE!r}."
)
print(f"Skipped {skipped_chunk_records} unchanged vectors already present in Pinecone.")


Upserted 753 vectors into 'fahmai-text-embedding-3-large' namespace 'knowledge-base-markdown'.
Skipped 0 unchanged vectors already present in Pinecone.


In [9]:
verification_id = embedded_records[0]["id"]
fetched = index.fetch(ids=[verification_id], namespace=NAMESPACE)
query_text = "ค่าจัดส่ง Express กรุงเทพ ราคาเท่าไหร่"
query_vector = embed_text_batch([query_text])[0]
matches = index.query(
    vector=query_vector, top_k=TOP_K, namespace=NAMESPACE, include_metadata=True
)

print(f"Fetched vector id: {verification_id}")
print(fetched)

match_rows = []
for match in getattr(matches, "matches", []):
    match_payload = match.to_dict() if hasattr(match, "to_dict") else match
    metadata = match_payload.get("metadata", {})
    match_rows.append(
        {
            "score": match_payload.get("score"),
            "id": match_payload.get("id"),
            "file_path": metadata.get("file_path"),
            "heading_path": " > ".join(metadata.get("heading_path", [])),
            "text_preview": metadata.get("text_preview"),
        }
    )
display(pd.DataFrame(match_rows))


Fetched vector id: knowledge_base/policies/cancellation_policy.md#chunk_0000
FetchResponse(namespace='knowledge-base-markdown', vectors={'knowledge_base/policies/cancellation_policy.md#chunk_0000': Vector(id='knowledge_base/policies/cancellation_policy.md#chunk_0000', values=[-0.0471423119, -0.0942208767, -0.0154192653, 0.0790486336, -0.00938702561, -0.0437317453, -0.0272048432, 0.070442535, -0.0211646352, 0.0210690126, 0.0149411485, -0.0142000681, 0.0430305079, -0.025228627, -0.0486085378, 0.0100962324, -0.0237145908, 0.0126302512, -0.000750543724, -0.0194434151, -0.0420424, 0.00723151583, 0.0377074741, -0.0390143283, -0.0127338432, 0.0205112081, -0.0378031, 0.013443049, -0.0334044248, 0.0208777655, 0.0285276324, 0.0600514635, -0.0600514635, -0.00996076595, 0.00877344236, 0.00790486392, 0.0433492512, -0.039301198, 0.0612626933, 0.0172122028, -0.00153694616, -0.0112118376, -0.00795267522, 0.0149331801, -0.0417236574, 0.00573341688, -0.0172281414, 0.00648645079, -0.0746499598, -0.059095

,score,id,file_path,heading_path,text_preview
0,0.642015,knowledge_base/policies/shipping_policy.md#chunk_0001,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง,# นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่\n## 2. ค่าจัดส่ง\n\n| มูลค่าคำสั่งซื้อ | ค่าจัดส่ง | หมายเหตุ |\n|----------------|----------|----------|\n| ฿500 ขึ้น...
1,0.624159,knowledge_base/policies/shipping_policy.md#chunk_0005,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 3. ระยะเวลาจัดส่ง > 3.2 การจัดส่ง Express (กรุงเทพฯ + ปริมณฑลเท่านั้น),# นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่\n## 3. ระยะเวลาจัดส่ง\n### 3.2 การจัดส่ง Express (กรุงเทพฯ + ปริมณฑลเท่านั้น)\n\nบริการ Express จัดส่งภายใน **1 วันทำก...
2,0.548391,knowledge_base/policies/shipping_policy.md#chunk_0002,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 2. ค่าจัดส่ง > 2.1 สมาชิก Gold และ Platinum,"# นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่\n## 2. ค่าจัดส่ง\n### 2.1 สมาชิก Gold และ Platinum\n\n- **Gold (ยอดสะสม ฿10,000–฿49,999):** จัดส่งฟรีทุกคำสั่งซื้อ ไม่..."
3,0.523609,knowledge_base/policies/return_policy.md#chunk_0016,knowledge_base/policies/return_policy.md,นโยบายการคืนสินค้าและการขอเงินคืน — ร้านฟ้าใหม่ > 7. ค่าใช้จ่ายในการจัดส่งคืน,# นโยบายการคืนสินค้าและการขอเงินคืน — ร้านฟ้าใหม่\n## 7. ค่าใช้จ่ายในการจัดส่งคืน\n\n| สาเหตุการคืน | ผู้รับผิดชอบค่าจัดส่ง | อัตราค่าจัดส่ง |\n|-----------...
4,0.520932,knowledge_base/policies/shipping_policy.md#chunk_0004,knowledge_base/policies/shipping_policy.md,นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่ > 3. ระยะเวลาจัดส่ง > 3.1 การจัดส่งมาตรฐาน,# นโยบายการจัดส่งสินค้า — ร้านฟ้าใหม่\n## 3. ระยะเวลาจัดส่ง\n### 3.1 การจัดส่งมาตรฐาน\n\n| พื้นที่จัดส่ง | ระยะเวลา (วันทำการ) |\n|---------------|---------...
